In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
import xarray as xr
import xgboost as xgb
import optuna

In [4]:
folder = Path(r"D:\INCOIS\Data_Float_Fixed")

files = sorted(folder.glob("*.nc"))

phyto_ds = xr.open_dataset(
    r"D:\INCOIS\PHYTOBLOOM_GOM.nc"
)

In [5]:
r=[]

In [6]:
for file in files:

    date = pd.to_datetime(
        file.name[1:9],
        format="%Y%m%d"
    )

    if date.year <= 2023:
        continue

    ds = xr.open_dataset(file)

    ds = ds.sel(
        lat=slice(11,7),
        lon=slice(78,81)
    )

    chlor = ds["CHLOR_A"].values.ravel()

    bloom = (
        phyto_ds["PHYTOBLOOM"]
        .sel(time=date)
        .values
        .ravel()
    )

    lat_grid, lon_grid = np.meshgrid(
    ds["lat"].values,
    ds["lon"].values,
    indexing="ij"
)

    temp = pd.DataFrame({
    "date": date,
    "LAT": lat_grid.ravel(),
    "LON": lon_grid.ravel(),
    "CHLOR_A": chlor,
    "PHYTOBLOOM": bloom
    })
    
    
    r.append(temp)

In [7]:
df_test = pd.concat(r, ignore_index=True)
print(len(df_test))
df_test

4817664


,date,LAT,LON,CHLOR_A,PHYTOBLOOM
0,2024-01-01,10.979163,78.020836,NaN,NaN
1,2024-01-01,10.979163,78.062500,NaN,NaN
2,2024-01-01,10.979163,78.104172,NaN,NaN
3,2024-01-01,10.979163,78.145836,NaN,NaN
4,2024-01-01,10.979163,78.187500,NaN,NaN
...,...,...,...,...,...
4817659,2025-11-30,7.020830,80.812500,NaN,NaN
4817660,2025-11-30,7.020830,80.854172,NaN,NaN
4817661,2025-11-30,7.020830,80.895836,NaN,NaN
4817662,2025-11-30,7.020830,80.937500,NaN,NaN


In [8]:
df_test = df_test.dropna()
len(df_test)

273283

In [9]:
df_test["date"] = pd.to_datetime(df_test["date"])

# Month
df_test["month"] = df_test["date"].dt.month

df_test["month_sin"] = np.sin(
    2 * np.pi * df_test["month"] / 12
)

df_test["month_cos"] = np.cos(
    2 * np.pi * df_test["month"] / 12
)

# Day of year
df_test["dayofyear"] = df_test["date"].dt.dayofyear

df_test["day_sin"] = np.sin(
    2 * np.pi * df_test["dayofyear"] / 365
)

df_test["day_cos"] = np.cos(
    2 * np.pi * df_test["dayofyear"] / 365
)

In [10]:
lat_min = df_test["LAT"].min()
lat_max = df_test["LAT"].max()

df_test["LAT_scaled"] = (
    2 *(df_test["LAT"] - lat_min)/(lat_max - lat_min)- 1
)

lon_min = df_test["LON"].min()
lon_max = df_test["LON"].max()

df_test["LON_scaled"] = (
    2 *(df_test["LON"] - lon_min)/(lon_max - lon_min)- 1
)

In [11]:
df_test.to_csv('test_gom(2024-25).csv')

In [12]:
rows = []

In [13]:
for file in files:

    date = pd.to_datetime(
        file.name[1:9],
        format="%Y%m%d"
    )

    if date.year > 2023:
        continue

    ds = xr.open_dataset(file)

    ds = ds.sel(
        lat=slice(11,7),
        lon=slice(78,81)
    )

    chlor = ds["CHLOR_A"].values.ravel()

    bloom = (
        phyto_ds["PHYTOBLOOM"]
        .sel(time=date)
        .values
        .ravel()
    )

    lat_grid, lon_grid = np.meshgrid(
    ds["lat"].values,
    ds["lon"].values,
    indexing="ij"
    )

    temp = pd.DataFrame({
    "date": date,
    "LAT": lat_grid.ravel(),
    "LON": lon_grid.ravel(),
    "CHLOR_A": chlor,
    "PHYTOBLOOM": bloom
    })

    
    rows.append(temp)

In [14]:
df = pd.concat(rows, ignore_index=True)
print(len(df))
df

52856064


,date,LAT,LON,CHLOR_A,PHYTOBLOOM
0,2003-01-01,10.979163,78.020836,NaN,NaN
1,2003-01-01,10.979163,78.062500,NaN,NaN
2,2003-01-01,10.979163,78.104172,NaN,NaN
3,2003-01-01,10.979163,78.145836,NaN,NaN
4,2003-01-01,10.979163,78.187500,NaN,NaN
...,...,...,...,...,...
52856059,2023-12-31,7.020830,80.812500,NaN,NaN
52856060,2023-12-31,7.020830,80.854172,NaN,NaN
52856061,2023-12-31,7.020830,80.895836,NaN,NaN
52856062,2023-12-31,7.020830,80.937500,NaN,NaN


In [15]:
df = df.dropna()
len(df)

4082861

In [16]:
df["date"] = pd.to_datetime(df["date"])

# Month
df["month"] = df["date"].dt.month

df["month_sin"] = np.sin(
    2 * np.pi * df["month"] / 12
)

df["month_cos"] = np.cos(
    2 * np.pi * df["month"] / 12
)

# Day of year
df["dayofyear"] = df["date"].dt.dayofyear

df["day_sin"] = np.sin(
    2 * np.pi * df["dayofyear"] / 365
)

df["day_cos"] = np.cos(
    2 * np.pi * df["dayofyear"] / 365
)

In [17]:
lat_min = df["LAT"].min()
lat_max = df["LAT"].max()

df["LAT_scaled"] = (
    2 *(df["LAT"] - lat_min)/(lat_max - lat_min)- 1
)

lon_min = df["LON"].min()
lon_max = df["LON"].max()
df["LON_scaled"] = (
    2 *(df["LON"] - lon_min)/(lon_max - lon_min)- 1
)

In [18]:
df.to_csv('training_gom(2003-2023).csv')

In [2]:
train_df = pd.read_csv("training_gom(2003-2023).csv")
test_df = pd.read_csv("test_gom(2024-25).csv")

print(train_df.shape)
print(test_df.shape)

(4082861, 14)
(273283, 14)


In [3]:
FEATURES = [
    "CHLOR_A",
    "day_sin",
    "day_cos",
    "month_sin",
    "month_cos",
    "LAT_scaled",
    "LON_scaled"
]

TARGET = "PHYTOBLOOM"

In [4]:
X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_test = test_df[FEATURES]
y_test = test_df[TARGET]

In [8]:

import joblib

final_model =  joblib.load('lgb_model_GOM.pkl')

final_model.fit(
    X_train,
    y_train
)

[LightGBM] [Warning] Unknown parameter: weight_mode
[LightGBM] [Warning] Unknown parameter: weight_mode
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.024363 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 928
[LightGBM] [Info] Number of data points in the train set: 4082861, number of used features: 7
[LightGBM] [Info] Start training from score -0.145816
[LightGBM] [Info] Start training from score -3.131013
[LightGBM] [Info] Start training from score -2.385858


,num_leaves,39
,max_depth,12
,learning_rate,0.02738388850765012
,n_estimators,60
,objective,'multiclass'
,min_split_gain,0.0012963994780004645
,min_child_samples,46
,subsample,0.8704588059140501
,colsample_bytree,0.8013089285041655
,reg_alpha,0.06198542335852684
,reg_lambda,0.004700171913470125


In [9]:
y_pred1 = final_model.predict(
    X_train
)
y_pred = final_model.predict(
    X_test
)

[LightGBM] [Warning] Unknown parameter: weight_mode
[LightGBM] [Warning] Unknown parameter: weight_mode


In [10]:
print("TRAIN RESULTS")
print(
    classification_report(
        y_train,
        y_pred1
    )
)

macro_f1 = f1_score(
    y_train,
    y_pred1,
    average="macro"
)

print(
    "Train Macro F1:",
    macro_f1
)
print()
print("TEST RESULTS")
print(
    classification_report(
        y_test,
        y_pred
    )
)

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro"
)

print(
    "Test Macro F1:",
    macro_f1
)

TRAIN RESULTS
              precision    recall  f1-score   support

         0.0       0.97      0.99      0.98   3528884
         1.0       0.83      0.61      0.70    178313
         2.0       0.88      0.83      0.85    375664

    accuracy                           0.96   4082861
   macro avg       0.89      0.81      0.85   4082861
weighted avg       0.96      0.96      0.96   4082861

Train Macro F1: 0.8455972920721865

TEST RESULTS
              precision    recall  f1-score   support

         0.0       0.97      0.99      0.98    232342
         1.0       0.69      0.67      0.68     12506
         2.0       0.86      0.76      0.81     28435

    accuracy                           0.95    273283
   macro avg       0.84      0.81      0.82    273283
weighted avg       0.95      0.95      0.95    273283

Test Macro F1: 0.8223642394415424


In [27]:
from xgboost import XGBClassifier

model = XGBClassifier(
    objective="multi:softmax",
    num_class=3,
    random_state=42
)

In [28]:
model.fit(
    X_train,
    y_train
)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softmax'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import lo

In [29]:
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

In [30]:
print("TRAIN RESULTS\n")

print(classification_report(y_train, train_pred))


print("Macro F1:",
      f1_score(y_train, train_pred, average='macro'))


print("\n\nTEST RESULTS\n")

print(classification_report(y_test, test_pred))


print("Macro F1:",
      f1_score(y_test, test_pred, average='macro'))

TRAIN RESULTS

              precision    recall  f1-score   support

         0.0       0.98      0.99      0.99   3528884
         1.0       0.82      0.71      0.76    178313
         2.0       0.88      0.87      0.88    375664

    accuracy                           0.97   4082861
   macro avg       0.90      0.86      0.87   4082861
weighted avg       0.97      0.97      0.97   4082861

Macro F1: 0.8743138662093388


TEST RESULTS

              precision    recall  f1-score   support

         0.0       0.98      0.98      0.98    232342
         1.0       0.66      0.72      0.69     12506
         2.0       0.86      0.79      0.82     28435

    accuracy                           0.95    273283
   macro avg       0.83      0.83      0.83    273283
weighted avg       0.95      0.95      0.95    273283

Macro F1: 0.8323818100684545


In [23]:
def objective_multi(trial):
    params = {
        "objective": "multi:softprob",
        "num_class": 3,

        "n_estimators": trial.suggest_int("n_estimators", 30, 300, step=10),
        "max_depth": trial.suggest_int("max_depth", 3, 14),  # narrowed hard around 4
        "learning_rate": trial.suggest_float("learning_rate", 0.10, 0.24, log=True),
        "subsample": trial.suggest_float("subsample", 0.70, 0.85),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 0.85),
        "min_child_weight": trial.suggest_int("min_child_weight", 18, 30),
        "gamma": trial.suggest_float("gamma", 0.05, 0.5, log=True),
        "lambda": trial.suggest_float("lambda", 0.4, 2.5, log=True),
        "alpha": trial.suggest_float("alpha", 1.7, 2.8),
        "grow_policy": trial.suggest_categorical("grow_policy", ["depthwise", "lossguide"]),

        "tree_method": "hist",
        "random_state": 42,
        "n_jobs": -1,
    }


    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    gaps, val_scores = [], []

    for train_idx, val_idx in skf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = xgb.XGBClassifier(**params)
        model.fit(X_tr, y_tr, verbose=False)

        train_f1 = f1_score(y_tr, model.predict(X_tr), average="macro")
        val_f1 = f1_score(y_val, model.predict(X_val), average="macro")

        val_scores.append(val_f1)
        gaps.append(train_f1 - val_f1)

    mean_val = np.mean(val_scores)
    mean_gap = np.mean(gaps)

    # Return a tuple: optuna will treat this as two objectives
    return mean_val, mean_gap

In [24]:
study = optuna.create_study(directions=["maximize", "minimize"]) 
study.optimize(objective_multi, n_trials=40, show_progress_bar=True)

[I 2026-07-16 10:42:12,935] A new study created in memory with name: no-name-298b271c-ec31-435a-9502-dcce235e3b00


  0%|          | 0/40 [00:00<?, ?it/s]

[I 2026-07-16 10:49:11,400] Trial 0 finished with values: [0.8803556255351564, 0.00527797937065515] and parameters: {'n_estimators': 150, 'max_depth': 8, 'learning_rate': 0.1977180514202269, 'subsample': 0.7950136458658352, 'colsample_bytree': 0.7589405890422465, 'min_child_weight': 23, 'gamma': 0.3951682011855159, 'lambda': 0.6481280260414632, 'alpha': 1.9284334369843046, 'grow_policy': 'depthwise'}.
[I 2026-07-16 11:06:39,028] Trial 1 finished with values: [0.8926044232521196, 0.017684263254462106] and parameters: {'n_estimators': 200, 'max_depth': 13, 'learning_rate': 0.14652397195818664, 'subsample': 0.7572092538692679, 'colsample_bytree': 0.7690703199670946, 'min_child_weight': 26, 'gamma': 0.13016623870199828, 'lambda': 1.5987038615192182, 'alpha': 2.0277849854472483, 'grow_policy': 'lossguide'}.
[I 2026-07-16 11:13:02,765] Trial 2 finished with values: [0.8852223115289322, 0.009846353639097472] and parameters: {'n_estimators': 120, 'max_depth': 12, 'learning_rate': 0.13594256036

In [45]:
best_trials = study.best_trials
i=0

for t in best_trials:
    print(f"{i} val_f1={t.values[0]:.4f}, gap={t.values[1]:.4f}, params={t.params}")
    i+=1

0 val_f1=0.8804, gap=0.0053, params={'n_estimators': 150, 'max_depth': 8, 'learning_rate': 0.1977180514202269, 'subsample': 0.7950136458658352, 'colsample_bytree': 0.7589405890422465, 'min_child_weight': 23, 'gamma': 0.3951682011855159, 'lambda': 0.6481280260414632, 'alpha': 1.9284334369843046, 'grow_policy': 'depthwise'}
1 val_f1=0.8926, gap=0.0177, params={'n_estimators': 200, 'max_depth': 13, 'learning_rate': 0.14652397195818664, 'subsample': 0.7572092538692679, 'colsample_bytree': 0.7690703199670946, 'min_child_weight': 26, 'gamma': 0.13016623870199828, 'lambda': 1.5987038615192182, 'alpha': 2.0277849854472483, 'grow_policy': 'lossguide'}
2 val_f1=0.8744, gap=0.0027, params={'n_estimators': 180, 'max_depth': 7, 'learning_rate': 0.1467313703829761, 'subsample': 0.8295196544886779, 'colsample_bytree': 0.713869298077358, 'min_child_weight': 26, 'gamma': 0.07180240697475329, 'lambda': 0.5207837566795598, 'alpha': 1.7829860859947315, 'grow_policy': 'lossguide'}
3 val_f1=0.8857, gap=0.00

In [9]:
def objective_multi(trial):
    # params = {
    #     "objective": "multi:softprob",
    #     "num_class": 3,

    #     "n_estimators": trial.suggest_int("n_estimators", 20, 90, step=5),
    #     "max_depth": trial.suggest_int("max_depth", 4, 9),
    #     "learning_rate": trial.suggest_float("learning_rate", 0.08, 0.25, log=True),
    #     "subsample": trial.suggest_float("subsample", 0.65, 0.85),
    #     "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 0.88),
    #     "min_child_weight": trial.suggest_int("min_child_weight", 15, 30),
    #     "gamma": trial.suggest_float("gamma", 0.02, 1.0, log=True),
    #     "lambda": trial.suggest_float("lambda", 0.3, 5.0, log=True),
    #     "alpha": trial.suggest_float("alpha", 1.6, 2.8),
    #     "grow_policy": trial.suggest_categorical("grow_policy", ["depthwise", "lossguide"]),

    #     "tree_method": "hist",
    #     "random_state": 42,
    #     "n_jobs": -1,
    # }
    params = {
        "objective": "multi:softprob",
        "num_class": 3,
        "n_estimators": trial.suggest_int("n_estimators", 40, 180, step=10),
        "max_depth": trial.suggest_int("max_depth", 5, 9),
        "learning_rate": trial.suggest_float("learning_rate", 0.10, 0.21, log=True),
        "subsample": trial.suggest_float("subsample", 0.70, 0.82),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 0.80),
        "min_child_weight": trial.suggest_int("min_child_weight", 18, 30),
        "gamma": trial.suggest_float("gamma", 0.08, 0.25, log=True),
        "lambda": trial.suggest_float("lambda", 0.4, 2.3, log=True),
        "alpha": trial.suggest_float("alpha", 1.7, 2.4),
        "grow_policy": trial.suggest_categorical("grow_policy", ["depthwise", "lossguide"]),
        "tree_method": "hist",
        "random_state": 42,
        "n_jobs": -1,
    }


    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    gaps, val_scores = [], []

    for train_idx, val_idx in skf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = xgb.XGBClassifier(**params)
        model.fit(X_tr, y_tr, verbose=False)

        train_f1 = f1_score(y_tr, model.predict(X_tr), average="macro")
        val_f1 = f1_score(y_val, model.predict(X_val), average="macro")

        val_scores.append(val_f1)
        gaps.append(train_f1 - val_f1)

    mean_val = np.mean(val_scores)
    mean_gap = np.mean(gaps)

    # Return a tuple: optuna will treat this as two objectives
    return mean_val, mean_gap

In [10]:
study = optuna.create_study(directions=["maximize", "minimize"]) 
study.optimize(objective_multi, n_trials=60, show_progress_bar=True)

[I 2026-07-20 11:19:45,774] A new study created in memory with name: no-name-a5799d9c-8d68-4228-abd7-d9dc45284fff


  0%|          | 0/60 [00:00<?, ?it/s]

[I 2026-07-20 11:24:41,560] Trial 0 finished with values: [0.8725807583855714, 0.0024926626362008707] and parameters: {'n_estimators': 120, 'max_depth': 7, 'learning_rate': 0.16516724609566746, 'subsample': 0.7907493768205527, 'colsample_bytree': 0.7184858951766596, 'min_child_weight': 18, 'gamma': 0.08561689041099817, 'lambda': 1.2794555368058267, 'alpha': 1.7911104661273747, 'grow_policy': 'lossguide'}.
[I 2026-07-20 11:32:09,668] Trial 1 finished with values: [0.8778419213122289, 0.00411847360690174] and parameters: {'n_estimators': 160, 'max_depth': 8, 'learning_rate': 0.14044812840025747, 'subsample': 0.7922564975932839, 'colsample_bytree': 0.7909041685848435, 'min_child_weight': 25, 'gamma': 0.23288975582432148, 'lambda': 0.6348280940525617, 'alpha': 1.8473957052567, 'grow_policy': 'depthwise'}.
[I 2026-07-20 11:39:38,185] Trial 2 finished with values: [0.8773076527002225, 0.004308052145651953] and parameters: {'n_estimators': 150, 'max_depth': 9, 'learning_rate': 0.1118169818492

In [12]:
best_trials = study.best_trials
i=0

for t in best_trials:
    print(f"{i} val_f1={t.values[0]:.4f}, gap={t.values[1]:.4f}, params={t.params}")
    i+=1

0 val_f1=0.8726, gap=0.0025, params={'n_estimators': 120, 'max_depth': 7, 'learning_rate': 0.16516724609566746, 'subsample': 0.7907493768205527, 'colsample_bytree': 0.7184858951766596, 'min_child_weight': 18, 'gamma': 0.08561689041099817, 'lambda': 1.2794555368058267, 'alpha': 1.7911104661273747, 'grow_policy': 'lossguide'}
1 val_f1=0.8778, gap=0.0041, params={'n_estimators': 160, 'max_depth': 8, 'learning_rate': 0.14044812840025747, 'subsample': 0.7922564975932839, 'colsample_bytree': 0.7909041685848435, 'min_child_weight': 25, 'gamma': 0.23288975582432148, 'lambda': 0.6348280940525617, 'alpha': 1.8473957052567, 'grow_policy': 'depthwise'}
2 val_f1=0.8651, gap=0.0011, params={'n_estimators': 180, 'max_depth': 5, 'learning_rate': 0.15536859475993905, 'subsample': 0.7037615151398394, 'colsample_bytree': 0.6797861959236203, 'min_child_weight': 30, 'gamma': 0.2410429639101309, 'lambda': 1.9215735885968859, 'alpha': 2.1751975697584385, 'grow_policy': 'depthwise'}
3 val_f1=0.8705, gap=0.001

In [14]:
params=best_trials[15].params
params

{'n_estimators': 50,
 'max_depth': 5,
 'learning_rate': 0.10520329882637908,
 'subsample': 0.7649532340563672,
 'colsample_bytree': 0.7783288114821567,
 'min_child_weight': 27,
 'gamma': 0.18520037091731034,
 'lambda': 1.290553674841062,
 'alpha': 2.135863324804945,
 'grow_policy': 'lossguide'}

In [16]:
model = xgb.XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        **params,
        tree_method="hist",
        random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
y_pred1 = model.predict(
    X_train
)

y_pred = model.predict(
    X_test
)
macro_f1_train = f1_score(
    y_train,
    y_pred1,
    average="macro"
)

macro_f1_test = f1_score(
    y_test,
    y_pred,
    average="macro"
)
gap=macro_f1_train-macro_f1_test
print(f" Gap: {gap} Train Macro F1 : {macro_f1_train}  Test Macro F1 : {macro_f1_test}")


 Gap: 0.02743037226727496 Train Macro F1 : 0.8460511151826982  Test Macro F1 : 0.8186207429154232


In [17]:
import joblib

# Save the model
joblib.dump(model, 'xgb_model_GOM.pkl')

['xgb_model_GOM.pkl']

In [13]:
i=0
for t in best_trials:
    params=t.params
    lgb_model = xgb.XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        **params,
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    lgb_model.fit(X_train, y_train)
    y_pred1 = lgb_model.predict(
        X_train
    )

    y_pred = lgb_model.predict(
        X_test
    )
    macro_f1_train = f1_score(
        y_train,
        y_pred1,
        average="macro"
    )

    macro_f1_test = f1_score(
        y_test,
        y_pred,
        average="macro"
    )
    gap=macro_f1_train-macro_f1_test
    print(f"{i} Gap: {gap} Train Macro F1 : {macro_f1_train}  Test Macro F1 : {macro_f1_test}")
    i+=1


0 Gap: 0.04205238343561035 Train Macro F1 : 0.8750081516084123  Test Macro F1 : 0.8329557681728019
1 Gap: 0.04789928065604088 Train Macro F1 : 0.8819468224359693  Test Macro F1 : 0.8340475417799285
2 Gap: 0.03560986535960087 Train Macro F1 : 0.866124474467886  Test Macro F1 : 0.8305146091082851
3 Gap: 0.038905394861400655 Train Macro F1 : 0.8721517428776909  Test Macro F1 : 0.8332463480162903
4 Gap: 0.04056496335841564 Train Macro F1 : 0.874460186343022  Test Macro F1 : 0.8338952229846064
5 Gap: 0.04480499862188192 Train Macro F1 : 0.8793237694140982  Test Macro F1 : 0.8345187707922163
6 Gap: 0.037235050765720756 Train Macro F1 : 0.8686041599721918  Test Macro F1 : 0.831369109206471
7 Gap: 0.04049971322899859 Train Macro F1 : 0.8740928564380953  Test Macro F1 : 0.8335931432090967
8 Gap: 0.058089556903123896 Train Macro F1 : 0.8899237888178183  Test Macro F1 : 0.8318342319146944
9 Gap: 0.03845899202784553 Train Macro F1 : 0.8704054784630588  Test Macro F1 : 0.8319464864352133
10 Gap: 0.

In [2]:
import re

# Paste your full best_trials output here (all lines, as printed)
log_text = """
0 val_f1=0.8762, gap=0.0044, params={'n_estimators': 100, 'max_depth': 10, 'learning_rate': 0.10944709489657091, 'subsample': 0.7317589604995012, 'colsample_bytree': 0.6953290771341716, 'min_child_weight': 30, 'gamma': 0.11387934964586463, 'lambda': 1.0549159239745896, 'alpha': 2.3238756182352653, 'grow_policy': 'lossguide'}
1 val_f1=0.8229, gap=0.0001, params={'n_estimators': 40, 'max_depth': 4, 'learning_rate': 0.11571025679159974, 'subsample': 0.8093042317648745, 'colsample_bytree': 0.7137796330429824, 'min_child_weight': 19, 'gamma': 0.06710552538885642, 'lambda': 1.0497518541996316, 'alpha': 2.3577904235627254, 'grow_policy': 'lossguide'}
2 val_f1=0.8791, gap=0.0049, params={'n_estimators': 150, 'max_depth': 8, 'learning_rate': 0.19951730034429815, 'subsample': 0.8153599430710333, 'colsample_bytree': 0.6637596467644058, 'min_child_weight': 30, 'gamma': 0.07668815448155591, 'lambda': 1.354161833159287, 'alpha': 2.161861867520875, 'grow_policy': 'depthwise'}
3 val_f1=0.8442, gap=0.0002, params={'n_estimators': 170, 'max_depth': 3, 'learning_rate': 0.11020927771886237, 'subsample': 0.7463897677486803, 'colsample_bytree': 0.6851800417789057, 'min_child_weight': 25, 'gamma': 0.20011385667234774, 'lambda': 0.9786677476354191, 'alpha': 2.477017163520082, 'grow_policy': 'lossguide'}
4 val_f1=0.8699, gap=0.0024, params={'n_estimators': 50, 'max_depth': 9, 'learning_rate': 0.13315616116707724, 'subsample': 0.7769848625826535, 'colsample_bytree': 0.7191176265720222, 'min_child_weight': 21, 'gamma': 0.14413392225983668, 'lambda': 2.2250365475502463, 'alpha': 2.3660049667675533, 'grow_policy': 'lossguide'}
5 val_f1=0.8919, gap=0.0164, params={'n_estimators': 150, 'max_depth': 12, 'learning_rate': 0.23058151531363136, 'subsample': 0.7065884308315366, 'colsample_bytree': 0.7768362240149912, 'min_child_weight': 25, 'gamma': 0.08358642799951045, 'lambda': 1.645467116437388, 'alpha': 2.2473019769861433, 'grow_policy': 'lossguide'}
6 val_f1=0.8569, gap=0.0005, params={'n_estimators': 160, 'max_depth': 4, 'learning_rate': 0.12045942038422806, 'subsample': 0.7167060585840063, 'colsample_bytree': 0.7906639393105257, 'min_child_weight': 21, 'gamma': 0.05741168609428335, 'lambda': 0.8437252135984831, 'alpha': 2.0966838301719943, 'grow_policy': 'lossguide'}
7 val_f1=0.8683, gap=0.0016, params={'n_estimators': 110, 'max_depth': 7, 'learning_rate': 0.12681834341496523, 'subsample': 0.7983176118838361, 'colsample_bytree': 0.6667188264418532, 'min_child_weight': 28, 'gamma': 0.226545400219929, 'lambda': 0.9493625065932157, 'alpha': 2.1581003973796196, 'grow_policy': 'lossguide'}
8 val_f1=0.8805, gap=0.0058, params={'n_estimators': 180, 'max_depth': 8, 'learning_rate': 0.18875251030281376, 'subsample': 0.7616511421803682, 'colsample_bytree': 0.7086357151625736, 'min_child_weight': 18, 'gamma': 0.07069723855505121, 'lambda': 0.7495944655229668, 'alpha': 2.3503658242522247, 'grow_policy': 'lossguide'}
9 val_f1=0.8755, gap=0.0033, params={'n_estimators': 140, 'max_depth': 7, 'learning_rate': 0.1919612183384838, 'subsample': 0.729443379320247, 'colsample_bytree': 0.7716040252146499, 'min_child_weight': 19, 'gamma': 0.11970646881789365, 'lambda': 0.4921155405678625, 'alpha': 1.7681970041454256, 'grow_policy': 'depthwise'}
10 val_f1=0.8825, gap=0.0066, params={'n_estimators': 200, 'max_depth': 9, 'learning_rate': 0.12265184576472421, 'subsample': 0.8252593884668175, 'colsample_bytree': 0.816588666106996, 'min_child_weight': 21, 'gamma': 0.11683458824067286, 'lambda': 0.629800092334488, 'alpha': 2.3184901734364347, 'grow_policy': 'depthwise'}
11 val_f1=0.8897, gap=0.0128, params={'n_estimators': 180, 'max_depth': 10, 'learning_rate': 0.22711416599737455, 'subsample': 0.7151551830136728, 'colsample_bytree': 0.8014104022716946, 'min_child_weight': 21, 'gamma': 0.14189286921056404, 'lambda': 1.0317780123124702, 'alpha': 2.149407651894446, 'grow_policy': 'depthwise'}
12 val_f1=0.8799, gap=0.0055, params={'n_estimators': 100, 'max_depth': 9, 'learning_rate': 0.20570893291292117, 'subsample': 0.8012349342310094, 'colsample_bytree': 0.8238108906510386, 'min_child_weight': 27, 'gamma': 0.22634627424816425, 'lambda': 1.6914520412510292, 'alpha': 2.3216086954665194, 'grow_policy': 'lossguide'}
13 val_f1=0.8510, gap=0.0003, params={'n_estimators': 130, 'max_depth': 3, 'learning_rate': 0.20257969756813482, 'subsample': 0.7199887402884849, 'colsample_bytree': 0.7278807399942958, 'min_child_weight': 18, 'gamma': 0.06093384356901913, 'lambda': 0.5475813918007015, 'alpha': 2.0826063197693667, 'grow_policy': 'lossguide'}
14 val_f1=0.8910, gap=0.0143, params={'n_estimators': 160, 'max_depth': 13, 'learning_rate': 0.15339484469299375, 'subsample': 0.7544755281964334, 'colsample_bytree': 0.8232719210676371, 'min_child_weight': 23, 'gamma': 0.3965895105648431, 'lambda': 2.2504881618094714, 'alpha': 2.326459971799494, 'grow_policy': 'lossguide'}
15 val_f1=0.8611, gap=0.0007, params={'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.14665598716607936, 'subsample': 0.7041616956257574, 'colsample_bytree': 0.8007085456788328, 'min_child_weight': 25, 'gamma': 0.07833113435533068, 'lambda': 0.4127296129319316, 'alpha': 2.1999696011638976, 'grow_policy': 'lossguide'}
16 val_f1=0.8502, gap=0.0003, params={'n_estimators': 110, 'max_depth': 3, 'learning_rate': 0.22731384282105221, 'subsample': 0.8305319204398793, 'colsample_bytree': 0.8320237657476159, 'min_child_weight': 19, 'gamma': 0.11905222284687504, 'lambda': 2.1731458966759907, 'alpha': 2.77880481722537, 'grow_policy': 'lossguide'}
17 val_f1=0.8950, gap=0.0227, params={'n_estimators': 180, 'max_depth': 14, 'learning_rate': 0.20117839890784578, 'subsample': 0.7818795536534398, 'colsample_bytree': 0.8487569418008967, 'min_child_weight': 27, 'gamma': 0.15766752470485199, 'lambda': 1.3520249934018735, 'alpha': 1.997677705715474, 'grow_policy': 'lossguide'}
18 val_f1=0.8576, gap=0.0006, params={'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.10364766048363648, 'subsample': 0.7809749647192663, 'colsample_bytree': 0.7908309290014222, 'min_child_weight': 30, 'gamma': 0.05287293229210929, 'lambda': 1.3814465698011986, 'alpha': 1.9791262458941516, 'grow_policy': 'lossguide'}
19 val_f1=0.8867, gap=0.0102, params={'n_estimators': 190, 'max_depth': 10, 'learning_rate': 0.18412580232521003, 'subsample': 0.8154561572651473, 'colsample_bytree': 0.6959134810191328, 'min_child_weight': 25, 'gamma': 0.14057402278902642, 'lambda': 0.6133168477196687, 'alpha': 2.7828429276405124, 'grow_policy': 'lossguide'}
20 val_f1=0.8642, gap=0.0015, params={'n_estimators': 90, 'max_depth': 7, 'learning_rate': 0.10939814502545417, 'subsample': 0.7711759130909477, 'colsample_bytree': 0.685405040150334, 'min_child_weight': 22, 'gamma': 0.16638958084244027, 'lambda': 0.8450269956315326, 'alpha': 2.3827941819596896, 'grow_policy': 'depthwise'}
21 val_f1=0.8832, gap=0.0091, params={'n_estimators': 40, 'max_depth': 13, 'learning_rate': 0.23389011437644536, 'subsample': 0.7387879475259895, 'colsample_bytree': 0.8391494659269236, 'min_child_weight': 25, 'gamma': 0.33190150746287034, 'lambda': 0.7084635752357078, 'alpha': 2.1049431206241453, 'grow_policy': 'lossguide'}
22 val_f1=0.8650, gap=0.0016, params={'n_estimators': 70, 'max_depth': 7, 'learning_rate': 0.15437206471266035, 'subsample': 0.7611226379026037, 'colsample_bytree': 0.6783016379944227, 'min_child_weight': 28, 'gamma': 0.10234733040856327, 'lambda': 0.5006541024694537, 'alpha': 2.3089851035723514, 'grow_policy': 'lossguide'}
23 val_f1=0.8901, gap=0.0136, params={'n_estimators': 170, 'max_depth': 12, 'learning_rate': 0.1337711569216201, 'subsample': 0.8003895366355729, 'colsample_bytree': 0.7562873506879672, 'min_child_weight': 22, 'gamma': 0.22143332980386596, 'lambda': 1.252124957764437, 'alpha': 1.7229309104252324, 'grow_policy': 'depthwise'}
24 val_f1=0.8811, gap=0.0060, params={'n_estimators': 170, 'max_depth': 9, 'learning_rate': 0.12876176667516329, 'subsample': 0.7938732306938991, 'colsample_bytree': 0.7741859773197596, 'min_child_weight': 24, 'gamma': 0.08022037505313662, 'lambda': 1.2001995093362452, 'alpha': 1.8929748712904222, 'grow_policy': 'depthwise'}
25 val_f1=0.8849, gap=0.0102, params={'n_estimators': 80, 'max_depth': 13, 'learning_rate': 0.11341025064273932, 'subsample': 0.8392670788821844, 'colsample_bytree': 0.7690051358944665, 'min_child_weight': 18, 'gamma': 0.28130984804064707, 'lambda': 2.345692539166496, 'alpha': 1.8201161520759477, 'grow_policy': 'lossguide'}
26 val_f1=0.8882, gap=0.0111, params={'n_estimators': 140, 'max_depth': 11, 'learning_rate': 0.17721090893209832, 'subsample': 0.8155179062550316, 'colsample_bytree': 0.7192516974525307, 'min_child_weight': 21, 'gamma': 0.26457821813619475, 'lambda': 1.4993574218413432, 'alpha': 2.6207755881420063, 'grow_policy': 'lossguide'}
""".strip()  # <-- replace with the FULL pasted block (all 27 lines)

pattern = r"(\d+) val_f1=([\d.]+), gap=([\d.]+), params=(\{.*?\})"
matches = re.findall(pattern, log_text)

trials = []
for num, val_f1, gap, params_str in matches:
    trials.append({
        "trial": int(num),
        "cv_val_f1": float(val_f1),
        "cv_gap": float(gap),
        "params": eval(params_str)
    })

print(f"Parsed {len(trials)} trials")

Parsed 27 trials


In [8]:
i=0
for t in trials:
    params=t["params"]
    lgb_model = xgb.XGBClassifier(
    objective='multiclass',
    num_class=3,
    **params,
    random_state=42
    )

    lgb_model.fit(X_train, y_train)
    y_pred1 = lgb_model.predict(
        X_train
    )

    y_pred = lgb_model.predict(
        X_test
    )
    macro_f1_train = f1_score(
        y_train,
        y_pred1,
        average="macro"
    )

    macro_f1_test = f1_score(
        y_test,
        y_pred,
        average="macro"
    )
    gap=macro_f1_train-macro_f1_test
    print(f"{i} Gap: {gap} Train Macro F1 : {macro_f1_train}  Test Macro F1 : {macro_f1_test}")
    i+=1


0 Gap: 0.049790522795544856 Train Macro F1 : 0.8804754627533914  Test Macro F1 : 0.8306849399578465
1 Gap: 0.028309487248564413 Train Macro F1 : 0.8232318723586296  Test Macro F1 : 0.7949223851100652
2 Gap: 0.05096859354536443 Train Macro F1 : 0.8836802603828086  Test Macro F1 : 0.8327116668374441
3 Gap: 0.028384462955062695 Train Macro F1 : 0.8446843739601805  Test Macro F1 : 0.8162999110051178
4 Gap: 0.0409147406745608 Train Macro F1 : 0.8723711495952241  Test Macro F1 : 0.8314564089206633
5 Gap: 0.08225389726848142 Train Macro F1 : 0.9083968139781522  Test Macro F1 : 0.8261429167096708
6 Gap: 0.03174109053454843 Train Macro F1 : 0.8573241589664011  Test Macro F1 : 0.8255830684318527
7 Gap: 0.038265792521352315 Train Macro F1 : 0.8694535509754232  Test Macro F1 : 0.8311877584540709
8 Gap: 0.05293950247704515 Train Macro F1 : 0.8857499945757134  Test Macro F1 : 0.8328104920986682
9 Gap: 0.04407606125170316 Train Macro F1 : 0.878344017781654  Test Macro F1 : 0.8342679565299509
10 Gap: 

In [ ]:
i=0
for t in best_trials:
    params=t.params
    lgb_model = xgb.XGBClassifier(
    objective='multiclass',
    num_class=3,
    **params,
    random_state=42
    )

    lgb_model.fit(X_train, y_train)
    y_pred1 = lgb_model.predict(
        X_train
    )

    y_pred = lgb_model.predict(
        X_test
    )
    macro_f1_train = f1_score(
        y_train,
        y_pred1,
        average="macro"
    )

    macro_f1_test = f1_score(
        y_test,
        y_pred,
        average="macro"
    )
    gap=macro_f1_train-macro_f1_test
    print(f"{i} Gap: {gap} Train Macro F1 : {macro_f1_train}  Test Macro F1 : {macro_f1_test}")
    i+=1


0 Gap: 0.05262558775782644 Train Macro F1 : 0.8854324203205097  Test Macro F1 : 0.8328068325626833
1 Gap: 0.08342420267633133 Train Macro F1 : 0.9102737889448292  Test Macro F1 : 0.8268495862684979
2 Gap: 0.043893066218946175 Train Macro F1 : 0.8768974398411326  Test Macro F1 : 0.8330043736221864
3 Gap: 0.06406841682260256 Train Macro F1 : 0.8941944821218916  Test Macro F1 : 0.830126065299289
4 Gap: 0.1011935326511898 Train Macro F1 : 0.9243613951883529  Test Macro F1 : 0.8231678625371631
5 Gap: 0.08024611828751071 Train Macro F1 : 0.9075758901069192  Test Macro F1 : 0.8273297718194085
6 Gap: 0.0836242694375735 Train Macro F1 : 0.9108267340578012  Test Macro F1 : 0.8272024646202277
7 Gap: 0.05985071864147817 Train Macro F1 : 0.8912609902156965  Test Macro F1 : 0.8314102715742183
8 Gap: 0.06132709361964883 Train Macro F1 : 0.8930262857474299  Test Macro F1 : 0.8316991921277811
9 Gap: 0.07602507871997699 Train Macro F1 : 0.9042740592060169  Test Macro F1 : 0.8282489804860399
10 Gap: 0.04

In [46]:
params=best_trials[16].params
params

{'n_estimators': 140,
 'max_depth': 3,
 'learning_rate': 0.10881987858257748,
 'subsample': 0.8446526339839784,
 'colsample_bytree': 0.8003642023896022,
 'min_child_weight': 20,
 'gamma': 0.2573469603906708,
 'lambda': 2.302483337281246,
 'alpha': 2.462096139775924,
 'grow_policy': 'depthwise'}